In [1]:
# naive approach
# C[m, n] = sum(A[m, k] * B[k, n] for k in range(K))

### The key ide: tile multiplication  
                 N
        ┌─────┬─────┬─────┐
      M │tile │tile │tile │
        ├─────┼─────┼─────┤
        │tile │tile │tile │
        └─────┴─────┴─────┘
each triton program instance computes one tile of output C  
to compute that tile: C tile += A tile * B tile

In [2]:
!pip install triton

In [3]:
!pip install torch

In [4]:
import torch

import triton
import triton.language as tl

In [5]:
@triton.jit
def matmul_k(
    a_ptr, # ptr to matrix a MxK
    b_ptr, # ptr to matrix b KxN
    c_ptr,# output matrix == MxN
    M: tl.constexpr,
    N: tl.constexpr,
    K: tl.constexpr,
    # (in memory even matrix is linear)
    # stride tells us how many mem.el. to skip when moving one position along tensor dimension
    stride_am,
    stride_ak,
    stride_bk,
    stride_bn,
    stride_cm,
    stride_cn,
    BLOCK_M: tl.constexpr,
    BLOCK_N: tl.constexpr,
    BLOCK_K: tl.constexpr,
):

    # pids: which tile does this program compute
    pid_m = tl.program_id(axis=0)
    pid_n = tl.program_id(axis=1)

    # row and col indices
    rows = pid_m * BLOCK_M + tl.arange(0, BLOCK_M)
    cols = pid_n * BLOCK_N + tl.arange(0, BLOCK_N)
    ks = tl.arange(0, BLOCK_K)

    # construct pointers to the current tiles
    a_ptrs = (
        a_ptr + rows[:, None] * stride_am + ks[None, :] * stride_ak )

    b_ptrs = (
        b_ptr + ks[:, None] * stride_bk + cols[None, :] * stride_bn )

    # accumulator: temporary memory space to store the running totals of dot prods
    accumulator = tl.zeros((BLOCK_M, BLOCK_N), dtype=tl.float32)

    for k_start in range(0, K, BLOCK_K):
        
        a = tl.load(
            a_ptrs,
            mask=(rows[:, None] < M) & (k_start + ks[None, :] < K),
            other=0.0, )

        b = tl.load(
            b_ptrs,
            mask=(k_start + ks[:, None] < K) & (cols[None, :] < N),
            other=0.0, )

        accumulator += tl.dot(a, b)

        # advance both pointers along k
        a_ptrs += BLOCK_K * stride_ak
        b_ptrs += BLOCK_K * stride_bk

    c_ptrs = (
        c_ptr
        + rows[:, None] * stride_cm
        + cols[None, :] * stride_cn )
    
    output_mask = ( 
        (rows[:, None] < M)
        & (cols[None, :] < N) )
    
    tl.store(c_ptrs, accumulator, mask=output_mask)

Now we will have a helper function

In [6]:
def matmul(a: torch.Tensor, b: torch.Tensor) -> torch.Tensor:
    # checking if anything is on its place
    assert a.ndim == 2 and b.ndim == 2
    assert a.shape[1] == b.shape[0]
    assert a.is_cuda and b.is_cuda
    assert a.dtype == torch.float16
    assert b.dtype == torch.float16

    M, K = a.shape
    _, N = b.shape

    c = torch.empty(
            (M, N),
            device=a.device,
            dtype=torch.float16, )

    BLOCK_M = 32
    BLOCK_N = 32
    BLOCK_K = 32
    
    grid = (
        triton.cdiv(M, BLOCK_M),
        triton.cdiv(N, BLOCK_N)
    )

    matmul_k[grid](
        a, b, c,
        M, N, K,
        a.stride(0), a.stride(1),
        b.stride(0), b.stride(1),
        c.stride(0), c.stride(1),
        BLOCK_M=BLOCK_M,
        BLOCK_N=BLOCK_N,
        BLOCK_K=BLOCK_K,
        num_warps=4, )

    return c

In [7]:
torch.manual_seed(0)

a = torch.randn(
        (128, 64),
        device='cuda',
        dtype=torch.float16, )
b = torch.randn(
        (64, 96),
        device='cuda',
        dtype=torch.float16, )

torch_test = torch.matmul(a, b)
triton_test = matmul(a, b)

In [8]:
print(torch.allclose(torch_test, triton_test, atol=1e-2, rtol=1e-2))
print("max error:", (triton_test - torch_test).abs().max())

True
max error: tensor(0.0078, device='cuda:0', dtype=torch.float16)


### Docs' MatMul  
Above matmul is a beginner level not so efficient, isn't optimized  
From here, implementing a matmul according to Triton docs

### L2 Cache Optimizations
"The order in which these blocks are computed does matter since it  
affects the L2 cache hit rate of our program and unfortunately, a simple 
row-major ordering isn't gonna cut it"

#### What is L2 Cache?  
L2 cache is a small, fast memory shared by the whole GPU  
> L2 cache remembers recently accessed GPU data so nearby kernels/programs can avoid repeatedly fetching it from slower GPU memory.

for example, in matrix multiplication, many output tiles reuse the same A or B tiles  
C₀₀ uses A₀ and B₀  
C₀₁ uses A₀ and B₁  ← A₀ reused  
C₁₀ uses A₁ and B₀  ← B₀ reused  

one possible solution is to launch blocks in an order that promotes data reuse  
this can be done by "SUPER-GROUPING" blocks in group of 'GROUP_M'  
rows before switching to the next column

In [9]:
#  Program ID
# pid = tl.program_id(axis=0)
#  Number of program ids along the M axis
# num_pid_m = tl.cdiv(M, BLOCK_SIZE_M)
#  Number of programs ids along the N axis
# num_pid_n = tl.cdiv(N, BLOCK_SIZE_N)
#  Number of programs in group
# num_pid_in_group = GROUP_SIZE_M * num_pid_n
#  Id of the group this program is in
# group_id = pid // num_pid_in_group
#  Row-id of the first program in the group
# first_pid_m = group_id * GROUP_SIZE_M
#  If `num_pid_m` isn't divisible by `GROUP_SIZE_M`, the last group is smaller
# group_size_m = min(num_pid_m - first_pid_m, GROUP_SIZE_M)
#  *Within groups*, programs are ordered in a column-major order
#  Row-id of the program in the *launch grid*
# pid_m = first_pid_m + ((pid % num_pid_in_group) % group_size_m)
#  Col-id of the program in the *launch grid*
# pid_n = (pid % num_pid_in_group) // group_size_m

![image](grouped_vs_row_major_ordering.png)

In [15]:
DEVICE = triton.runtime.driver.active.get_active_torch_device()


def is_cuda():
    return triton.runtime.driver.active.get_current_target().backend == "cuda"


def get_cuda_autotune_config():
    return [
        triton.Config({'BLOCK_SIZE_M': 128, 'BLOCK_SIZE_N': 256, 'BLOCK_SIZE_K': 64, 'GROUP_SIZE_M': 8}, num_stages=3,
                      num_warps=8),
        triton.Config({'BLOCK_SIZE_M': 64, 'BLOCK_SIZE_N': 256, 'BLOCK_SIZE_K': 32, 'GROUP_SIZE_M': 8}, num_stages=4,
                      num_warps=4),
        triton.Config({'BLOCK_SIZE_M': 128, 'BLOCK_SIZE_N': 128, 'BLOCK_SIZE_K': 32, 'GROUP_SIZE_M': 8}, num_stages=4,
                      num_warps=4),
        triton.Config({'BLOCK_SIZE_M': 128, 'BLOCK_SIZE_N': 64, 'BLOCK_SIZE_K': 32, 'GROUP_SIZE_M': 8}, num_stages=4,
                      num_warps=4),
        triton.Config({'BLOCK_SIZE_M': 64, 'BLOCK_SIZE_N': 128, 'BLOCK_SIZE_K': 32, 'GROUP_SIZE_M': 8}, num_stages=4,
                      num_warps=4),
        triton.Config({'BLOCK_SIZE_M': 128, 'BLOCK_SIZE_N': 32, 'BLOCK_SIZE_K': 32, 'GROUP_SIZE_M': 8}, num_stages=4,
                      num_warps=4),
        triton.Config({'BLOCK_SIZE_M': 64, 'BLOCK_SIZE_N': 32, 'BLOCK_SIZE_K': 32, 'GROUP_SIZE_M': 8}, num_stages=5,
                      num_warps=2),
        triton.Config({'BLOCK_SIZE_M': 32, 'BLOCK_SIZE_N': 64, 'BLOCK_SIZE_K': 32, 'GROUP_SIZE_M': 8}, num_stages=5,
                      num_warps=2),
        # Good config for fp8 inputs.
        triton.Config({'BLOCK_SIZE_M': 128, 'BLOCK_SIZE_N': 256, 'BLOCK_SIZE_K': 128, 'GROUP_SIZE_M': 8}, num_stages=3,
                      num_warps=8),
        triton.Config({'BLOCK_SIZE_M': 256, 'BLOCK_SIZE_N': 128, 'BLOCK_SIZE_K': 128, 'GROUP_SIZE_M': 8}, num_stages=3,
                      num_warps=8),
        triton.Config({'BLOCK_SIZE_M': 256, 'BLOCK_SIZE_N': 64, 'BLOCK_SIZE_K': 128, 'GROUP_SIZE_M': 8}, num_stages=4,
                      num_warps=4),
        triton.Config({'BLOCK_SIZE_M': 64, 'BLOCK_SIZE_N': 256, 'BLOCK_SIZE_K': 128, 'GROUP_SIZE_M': 8}, num_stages=4,
                      num_warps=4),
        triton.Config({'BLOCK_SIZE_M': 128, 'BLOCK_SIZE_N': 128, 'BLOCK_SIZE_K': 128, 'GROUP_SIZE_M': 8}, num_stages=4,
                      num_warps=4),
        triton.Config({'BLOCK_SIZE_M': 128, 'BLOCK_SIZE_N': 64, 'BLOCK_SIZE_K': 64, 'GROUP_SIZE_M': 8}, num_stages=4,
                      num_warps=4),
        triton.Config({'BLOCK_SIZE_M': 64, 'BLOCK_SIZE_N': 128, 'BLOCK_SIZE_K': 64, 'GROUP_SIZE_M': 8}, num_stages=4,
                      num_warps=4),
        triton.Config(
    {
        "BLOCK_SIZE_M": 32,
        "BLOCK_SIZE_N": 32,
        "BLOCK_SIZE_K": 32,
        "GROUP_SIZE_M": 1,
    },
    num_stages=3,
    num_warps=4,
),

triton.Config(
    {
        "BLOCK_SIZE_M": 32,
        "BLOCK_SIZE_N": 32,
        "BLOCK_SIZE_K": 32,
        "GROUP_SIZE_M": 8,
    },
    num_stages=3,
    num_warps=4,
),
        triton.Config({'BLOCK_SIZE_M': 128, 'BLOCK_SIZE_N': 32, 'BLOCK_SIZE_K': 64, 'GROUP_SIZE_M': 8}, num_stages=4,
                      num_warps=4)
    ]


def get_hip_autotune_config():
    sizes = [
        {'BLOCK_SIZE_M': 32, 'BLOCK_SIZE_N': 32, 'BLOCK_SIZE_K': 64, 'GROUP_SIZE_M': 6},
        {'BLOCK_SIZE_M': 64, 'BLOCK_SIZE_N': 32, 'BLOCK_SIZE_K': 64, 'GROUP_SIZE_M': 4},
        {'BLOCK_SIZE_M': 32, 'BLOCK_SIZE_N': 64, 'BLOCK_SIZE_K': 64, 'GROUP_SIZE_M': 6},
        {'BLOCK_SIZE_M': 64, 'BLOCK_SIZE_N': 64, 'BLOCK_SIZE_K': 64, 'GROUP_SIZE_M': 6},
        {'BLOCK_SIZE_M': 128, 'BLOCK_SIZE_N': 64, 'BLOCK_SIZE_K': 64, 'GROUP_SIZE_M': 4},
        {'BLOCK_SIZE_M': 128, 'BLOCK_SIZE_N': 128, 'BLOCK_SIZE_K': 64, 'GROUP_SIZE_M': 4},
        {'BLOCK_SIZE_M': 256, 'BLOCK_SIZE_N': 128, 'BLOCK_SIZE_K': 64, 'GROUP_SIZE_M': 4},
        {'BLOCK_SIZE_M': 256, 'BLOCK_SIZE_N': 256, 'BLOCK_SIZE_K': 64, 'GROUP_SIZE_M': 6},
    ]
    return [triton.Config(s | {'matrix_instr_nonkdim': 16}, num_warps=8, num_stages=2) for s in sizes]


def get_autotune_config():
    if is_cuda():
        return get_cuda_autotune_config()
    else:
        return get_hip_autotune_config()


# `triton.jit`'ed functions can be auto-tuned by using the `triton.autotune` decorator, which consumes:
#   - A list of `triton.Config` objects that define different configurations of
#       meta-parameters (e.g., `BLOCK_SIZE_M`) and compilation options (e.g., `num_warps`) to try
#   - An auto-tuning *key* whose change in values will trigger evaluation of all the
#       provided configs
@triton.autotune(
    configs=get_autotune_config(),
    key=['M', 'N', 'K'],
)
@triton.jit
def matmul_kernel(
    a_ptr, b_ptr, c_ptr, # pointer to matrices
    M, N, K, # dimensions
    # The stride variables represent how much to increase the ptr by when moving by 1
    # element in a particular dimension. E.g. `stride_am` is how much to increase `a_ptr`
    # by to get the element one row down (A has M rows).
    stride_am, stride_ak,
    stride_bk, stride_bn,
    stride_cm, stride_cn,
    BLOCK_SIZE_M: tl.constexpr, BLOCK_SIZE_N: tl.constexpr, BLOCK_SIZE_K: tl.constexpr,
    GROUP_SIZE_M: tl.constexpr,
    ACTIVATION: tl.constexpr
):
    pid = tl.program_id(axis=0)
    num_pid_m = tl.cdiv(M, BLOCK_SIZE_M)
    num_pid_n = tl.cdiv(N, BLOCK_SIZE_N)
    num_pid_in_group = GROUP_SIZE_M * num_pid_n
    group_id = pid // num_pid_in_group
    first_pid_m = group_id * GROUP_SIZE_M
    group_size_m = min(num_pid_m - first_pid_m, GROUP_SIZE_M)
    pid_m = first_pid_m + ((pid % num_pid_in_group) % group_size_m)
    pid_n = (pid % num_pid_in_group) // group_size_m

    # Add some integer bound assumptions.
    # This helps to guide integer analysis in the backend to optimize
    # load/store offset address calculation
    tl.assume(pid_m >= 0)
    tl.assume(pid_n >= 0)
    tl.assume(stride_am > 0)
    tl.assume(stride_ak > 0)
    tl.assume(stride_bn > 0)
    tl.assume(stride_bk > 0)
    tl.assume(stride_cm > 0)
    tl.assume(stride_cn > 0)

    offs_am = (pid_m * BLOCK_SIZE_M + tl.arange(0, BLOCK_SIZE_M)) % M
    offs_bn = (pid_n * BLOCK_SIZE_N + tl.arange(0, BLOCK_SIZE_N)) % N
    offs_ks = tl.arange(0, BLOCK_SIZE_K)

    a_ptrs = a_ptr + (offs_am[:, None] * stride_am + offs_ks[None, :] * stride_ak)
    b_ptrs = b_ptr + (offs_ks[:, None] * stride_bk + offs_bn[None, :] * stride_bn)


    accumulator = tl.zeros((BLOCK_SIZE_M, BLOCK_SIZE_N), dtype=tl.float32)
    for k in range(0, tl.cdiv(K, BLOCK_SIZE_K)):

        a =  tl.load(a_ptrs, mask=offs_ks[None, :] < K - k * BLOCK_SIZE_K, other=0.0)
        b = tl.load(b_ptrs, mask=offs_ks[:, None] < K - k * BLOCK_SIZE_K, other=0.0)

        accumulator = tl.dot(a, b, accumulator)

        a_ptrs += BLOCK_SIZE_K * stride_ak
        b_ptrs += BLOCK_SIZE_K * stride_bk

    # fuse(combine ops into one GPU kernel) arbitrary activation functions here
    # This avoids writing C to GPU memory and loading it again for a separate activation kernel, 
    # saving memory traffic and launch overhead.
    if ACTIVATION == 'leaky_relu':
        accumulator = leaky_relu(accumulator)
    c = accumulator.to(tl.float16)

    offs_cm = pid_m * BLOCK_SIZE_M + tl.arange(0, BLOCK_SIZE_M)
    offs_cn = pid_n * BLOCK_SIZE_N + tl.arange(0, BLOCK_SIZE_N)
    c_ptrs = c_ptr + (offs_cm[:, None] * stride_cm + offs_cn[None, :] * stride_cn)
    c_mask = (offs_cm[:, None] < M) & (offs_cn[None, :] < N)
    tl.store(c_ptrs, c, mask=c_mask)

# (?)We can fuse `leaky_relu` by providing it as an `ACTIVATION` meta-parameter in `matmul_kernel`.
@triton.jit
def leaky_relu(x):
    return tl.where(x >= 0, x, 0.01 * x)

Wrapper/Helper Function

In [16]:
def matmul_2(a, b, activation=""):
    # check
    assert a.shape[1] == b.shape[0] # dimensions
    assert a.is_contiguous() # matrix a contiguous
    M, K = a.shape
    K, N = b.shape

    # allocate output
    c = torch.empty((M, N), device=a.device, dtype=torch.float16)

    # 1D launch kernel where each block gets its own worker
    grid = lambda META: (triton.cdiv(M, META['BLOCK_SIZE_M']) 
                         * triton.cdiv(N, META['BLOCK_SIZE_N']), )

    matmul_kernel[grid](
        a, b, c,
        M, N, K,
        a.stride(0), a.stride(1),
        b.stride(0), b.stride(1),
        c.stride(0), c.stride(1),
        ACTIVATION=activation,
    )
    return c

In [17]:
# unit test
torch.manual_seed(0)
a = torch.rand((512, 512), device=DEVICE, dtype=torch.float16) - 0.5
b = torch.rand((512, 512), device=DEVICE, dtype=torch.float16) - 0.5

triton_out = matmul_2(a, b)
torch_out = torch.matmul(a, b)

print(f"triton output with fp16 inputs: {triton_out}")
print(f"torch output with fp16 inputs: {torch_out}")

if torch.allclose(triton_out, torch_out, atol=1e-2, rtol=0):
    print("match")
else:
    print("diff")

TORCH_HAS_FP8 = hasattr(torch, "float8_e5m2")
if TORCH_HAS_FP8 and is_cuda():
    torch.manual_seed(0)
    a = torch.randn((512, 512), device=DEVICE, dtype=torch.float16)
    b = torch.randn((512, 512), device=DEVICE, dtype=torch.float16)
    a = a.to(torch.float8_e5m2)
    # pre-transpose b for efficiency.
    b = b.T
    b = b.to(torch.float8_e5m2)
    triton_out = matmul_2(a, b)
    torch_out = torch.matmul(a.to(torch.float16), b.to(torch.float16))
    print(f"triton_output_with_fp8_inputs={triton_out}")
    print(f"torch_output_with_fp8_inputs={torch_out}")
    if torch.allclose(triton_out, torch_out, atol=0.125, rtol=0):
        print("match")
    else:
        print("diff")

triton output with fp16 inputs: tensor([[-0.5210,  2.8730, -1.0684,  ..., -0.6885,  0.5825,  0.4065],
        [ 2.3477, -1.5615,  1.9453,  ..., -0.5010, -1.8594, -2.7246],
        [ 0.3354, -1.3828,  2.9043,  ..., -0.8320, -1.8623,  3.4531],
        ...,
        [ 2.0586,  1.3125, -4.1484,  ..., -2.4297,  1.2734, -1.3037],
        [-0.3135, -2.9375, -1.8770,  ...,  1.1973, -1.7500,  4.0312],
        [ 0.3804, -0.9829,  0.4966,  ...,  1.4756,  0.3811,  2.7070]],
       device='cuda:0', dtype=torch.float16)
torch output with fp16 inputs: tensor([[-0.5210,  2.8730, -1.0684,  ..., -0.6885,  0.5825,  0.4065],
        [ 2.3477, -1.5615,  1.9453,  ..., -0.5010, -1.8594, -2.7246],
        [ 0.3354, -1.3828,  2.9043,  ..., -0.8320, -1.8623,  3.4531],
        ...,
        [ 2.0586,  1.3125, -4.1484,  ..., -2.4297,  1.2734, -1.3037],
        [-0.3135, -2.9375, -1.8770,  ...,  1.1973, -1.7500,  4.0312],
        [ 0.3804, -0.9829,  0.4966,  ...,  1.4756,  0.3811,  2.7070]],
       device='cuda:0', 

### Benchmark
#### Square Matrix Performance
compare the performance of this kernel against that of  
cuBLAS or rocBLAS  
focusing on square matrix